# NSD 教学子集：图像、caption、fMRI 的数据变化

先完成 NSD Data Access Agreement，再运行下载脚本。本 notebook 不会自动下载数据、模型或安装包。

两条路线在同一 NSD ID 对齐：

- PNG → RGB uint8 → 裁剪/缩放 → float32 → 图像模型输入（本节不训练图像网络）。
- caption → 句向量 → 每图 caption 均值；fMRI beta → 每 session z-score → 按图像重复求均值 → 回归预测句向量 → 小候选集检索。

注意：beta 已是官方 GLM 处理后的响应，不是原始 BOLD 时间序列；脑响应也不是天然的语义 embedding。下面是教学回归，不复现论文指标，也不做生成式图像重建。

In [ ]:
from pathlib import Path
import os
import ast, json, hashlib, subprocess, sys, importlib.metadata as im
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import zscore
import nibabel as nb

# 在仓库根目录或本 notebook 所在目录启动均可。
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/nsd_visuo_semantics').is_dir())
HERE = ROOT/'tutorials/nsd_minimal_walkthrough'
DATA = Path(os.environ.get('NSD_TEACHING_DATA_DIR', '/workspace/datasets/nsd_teaching_b' if os.name != 'nt' else str(HERE/'data')))
OUTPUT = Path(os.environ.get('NSD_TEACHING_OUTPUT_DIR', '/workspace/results/nsd_teaching_b' if os.name != 'nt' else str(HERE/'outputs')))
SEED = 20261008
VERTICES_PER_HEMISPHERE = 512   # 教学抽样，不是解剖学 ROI
RUN_FULL_GET_BETAS = False     # 原函数先读全脑，可能需要数 GB 内存
RUN_TEXT_MODEL = False         # True 需要你事先在服务器准备模型
RUN_TENSORFLOW = False         # True 需要安装 TensorFlow
LOCAL_MODEL = Path(os.environ.get('NSD_TEACHING_MODEL_DIR', '/workspace/checkpoints/nsd_teaching_b/sentence_model' if os.name != 'nt' else str(HERE/'models/sentence_model'))) # 必须是完整本地模型目录
report = {'seed': SEED, 'python': sys.version, 'arrays': {}, 'parameters': {}}
try:
    report['git_commit'] = subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()
except Exception:
    report['git_commit'] = 'unavailable'
for package in ['numpy','pandas','scipy','Pillow','nibabel','sentence-transformers','tensorflow','fracridge']:
    try: report.setdefault('versions', {})[package] = im.version(package)
    except im.PackageNotFoundError: pass
print('仓库:', ROOT)
print('Python:', sys.executable)

## 1. 清单就是两个机器之间的数据契约

首次在服务器运行 `--init` 生成 `DATA/data_manifest.json`：记录 session、图像 ID、caption、文件大小、ETag 和下载后的 SHA-256。之后续传或重用都不再 `--init`。清单保存在数据目录，不因普通 git pull 丢失；如需跨机器复用，单独同步这一小文件并比较哈希。先按 README 完成下载及 `--verify`。

In [ ]:
manifest_path = DATA/'data_manifest.json'
assert manifest_path.exists(), '尚未下载：完成协议后先运行 README 中的 --init --download 命令'
m = json.loads(manifest_path.read_text(encoding='utf-8'))
report['manifest_sha256'] = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
SUBJECT = m['subject']
N_SESSIONS = len(m['sessions'])
assert m['sessions'] == list(range(1, N_SESSIONS+1))
selected = pd.DataFrame(m['selected']).sort_values('nsd_id').reset_index(drop=True)
ids = selected.nsd_id.to_numpy(dtype=int)
print('图像数:', len(ids), '总下载 GB:', sum(f['size'] for f in m['files'])/1e9)
display(selected[['nsd_id','caption_index','trial_count','image_key']].head())

def describe(name, x):
    x = np.asarray(x)
    finite = x[np.isfinite(x)]
    info = {'shape': list(x.shape), 'dtype': str(x.dtype), 'bytes': x.nbytes,
            'finite_fraction': float(np.isfinite(x).mean()),
            'min': float(finite.min()) if finite.size else None,
            'max': float(finite.max()) if finite.size else None,
            'mean': float(finite.mean()) if finite.size else None}
    report['arrays'][name] = info
    print(name, info)
    return x

## 2. 读取真正的完整图像文件

选中图像的 PNG 完整保存于 `data/nsddata/stimuli/nsd/shared1000/`，不是缩略图；它是实验呈现的 NSD 裁剪图，不是未裁剪 COCO 原照片。用读取结果确认分辨率，不假定 PNG 的存储尺寸。

`73KID` 是从 1 开始的 NSD ID；caption 表用 `ID - 1`。不要把 COCO ID、shared 序号、NSD ID 混为一谈。

In [ ]:
image_paths = [DATA/x for x in selected.image_key]
raw_images = np.stack([np.asarray(Image.open(p).convert('RGB')) for p in image_paths])
describe('PNG_RGB_uint8', raw_images)
fig, axes = plt.subplots(1, min(4, len(ids)), figsize=(12, 3), squeeze=False)
for ax, i in zip(axes.ravel(), range(min(4,len(ids)))):
    ax.imshow(raw_images[i]); ax.set_title(f'NSD {ids[i]}'); ax.axis('off')
plt.show()
print('第一张 caption:', selected.captions.iloc[0])

## 3. 看清缩放、归一化和随机裁剪

先用 Pillow 做确定性缩放方便比较；再可选调用仓库实际的 `preprocess_batch_imgs` 与 `normalize`。参数 `image_size` 改变空间维度，batch_size 必须与实际批大小一致。仓库这个预处理函数总会随机裁剪；关闭后续 augment 不等于关闭该裁剪。归一化前必须确认数据是 0–255 还是 0–1，不能重复除以 255。

为了不触发原包的无关顶层依赖，`load_functions` 用 AST 从指定源码提取原函数定义，未改写函数体；依赖明确注入。它只适合这些独立教学函数，不是完整包导入的替代品。

In [ ]:
def load_functions(relative_path, names, namespace):
    path = ROOT/relative_path
    tree = ast.parse(path.read_text(encoding='utf-8-sig'))
    nodes = [n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name in names]
    assert {n.name for n in nodes} == set(names)
    exec(compile(ast.Module(body=nodes, type_ignores=[]), str(path), 'exec'), namespace)
    return namespace

for size in [128, 224]:
    resized = np.stack([np.asarray(Image.fromarray(x).resize((size,size), Image.Resampling.BILINEAR)) for x in raw_images[:4]])
    describe(f'deterministic_resize_{size}', resized.astype(np.float32)/255)

if RUN_TENSORFLOW:
    import tensorflow as tf
    image_ns = load_functions('src/nsd_visuo_semantics/get_dnn_activities/dataset_loader/tf_dataset_helper_functions.py',
                             ['preprocess_batch_imgs', 'normalize'], {'tf': tf})
    batch = raw_images[:4]
    tf.random.set_seed(SEED)
    hp = {'batch_size': len(batch), 'image_size': 224}
    processed = image_ns['preprocess_batch_imgs'](tf.constant(batch), hp)
    describe('repository_random_crop_224', processed.numpy())
    describe('repository_minus1_plus1', image_ns['normalize'](processed, '[-1,1]').numpy())
    fig, ax = plt.subplots(1,2,figsize=(7,3))
    ax[0].imshow(batch[0]); ax[0].set_title('完整输入')
    ax[1].imshow(processed.numpy()[0]); ax[1].set_title('原函数随机裁剪后')
    plt.show()
else:
    print('TensorFlow 原函数阶段未运行；先观察上面的确定性对照。')

## 4. 调用原代码读取 trial → 图像 ID

源码：`utils/nsd_get_data_light.py` 的 `read_behavior`、`get_conditions`、`get_subject_conditions`。注意 `read_behavior` 的注释写从 0 起，但函数实际按 TSV 的 SESSION 过滤，本项目调用是从 1 起。

前 6 个 session 并不保证每幅图像都有 3 次重复。比较 `keep_only_3repeats=False/True` 会改变哪些 ID 参与平均，而不是自动补齐数据。

In [ ]:
ns = load_functions('src/nsd_visuo_semantics/utils/nsd_get_data_light.py',
                    ['read_behavior','get_conditions','get_subject_conditions','average_over_conditions','get_betas'],
                    {'os': __import__('os'), 'np': np, 'pd': pd, 'nb': nb, 'zscore': zscore})
conditions_by_session = ns['get_conditions'](str(DATA), SUBJECT, N_SESSIONS)
conditions, conditions_sampled, all_ids = ns['get_subject_conditions'](str(DATA), SUBJECT, N_SESSIONS, keep_only_3repeats=False)
_, _, ids_three = ns['get_subject_conditions'](str(DATA), SUBJECT, N_SESSIONS, keep_only_3repeats=True)
assert set(ids).issubset(set(all_ids))
print('全部唯一 ID:', len(all_ids), '有恰好3次重复的 ID:', len(ids_three))
print('所选图像重复次数分布:', selected.trial_count.value_counts().to_dict())
display(ns['read_behavior'](str(DATA), SUBJECT, 1).head())

## 5. fMRI 文件 → 顶点×trial → z-score

源码 `get_betas` 在 fsaverage 空间把 lh/rh 拼接，再对每个顶点沿 trial 维做 z-score。下面先用 nibabel 的文件代理按固定步长读约 512 个顶点/半球，避免把全部 327,684 个顶点读进内存；不是 ROI，也不是原函数的全脑加载。逐顶点 z-score 与先全脑再取相同顶点的数学操作一致。

这里保留原函数的**按 session 使用全部 trial 做标准化**以讲解已有数据流；严谨建模时应另外评估标准化造成的信息泄漏，采用训练集统计量。本教学指标不能当论文结论。

In [ ]:
raw_sessions, z_sessions, vertex_ids = [], [], None
for session, trial_ids in zip(m['sessions'], conditions_by_session):
    hemis, hemis_ids = [], []
    offset = 0
    for hemisphere in ['lh', 'rh']:
        path = DATA/f'nsddata_betas/ppdata/{SUBJECT}/fsaverage/betas_fithrf_GLMdenoise_RR/{hemisphere}.betas_session{session:02d}.mgh'
        img = nb.load(str(path), mmap=True)
        stride = int(np.ceil(img.shape[0]/VERTICES_PER_HEMISPHERE))
        chosen_vertices = np.arange(0,img.shape[0],stride)
        arr = np.asarray(img.dataobj[::stride, :, :, :], dtype=np.float32).reshape(len(chosen_vertices), -1)
        assert arr.shape[1] == len(trial_ids), 'beta trial 与行为记录不一致'
        hemis.append(arr); hemis_ids.append(chosen_vertices+offset)
        offset += img.shape[0]
    raw = np.vstack(hemis)
    current_ids = np.concatenate(hemis_ids)
    if vertex_ids is None: vertex_ids = current_ids
    else: assert np.array_equal(vertex_ids, current_ids)
    raw_sessions.append(raw)
    z_sessions.append(zscore(raw, axis=-1).astype(np.float32))
raw = describe('sampled_raw_beta_vertices_by_trials', np.concatenate(raw_sessions, axis=-1))
standardized = describe('session_zscored_vertices_by_trials', np.concatenate(z_sessions, axis=-1))
print('首个顶点均值/标准差:', np.nanmean(z_sessions[0][0]), np.nanstd(z_sessions[0][0]))
# 只留下每次都有有限值的非恒定顶点，不能把 NaN 当成信号。
valid_vertices = np.isfinite(standardized).all(axis=1)
assert valid_vertices.any(), '抽样顶点无有效响应，请增加抽样数量'
standardized = standardized[valid_vertices]
vertex_ids = vertex_ids[valid_vertices]
if RUN_FULL_GET_BETAS:
    mask = np.zeros(offset, dtype=bool); mask[vertex_ids] = True
    full_one = ns['get_betas'](str(DATA), SUBJECT, 1, mask=mask, targetspace='fsaverage')[0]
    np.testing.assert_allclose(full_one, z_sessions[0][valid_vertices], atol=1e-5, rtol=1e-5)
    print('原 get_betas 与低内存读取一致')
report['parameters'].update({'sessions': m['sessions'], 'vertices_per_hemisphere_target': VERTICES_PER_HEMISPHERE,
                             'valid_vertex_ids': vertex_ids.tolist()})

## 6. 重复 trial → 每张图像的平均脑响应

调用原 `average_over_conditions(data, conditions, conditions_to_avg)`。输入是 `[顶点, trial]`，输出是 `[顶点, 所选图像]`；函数内部 `np.unique` 会排序，所以 caption 与图像也必须按同一 ID 顺序排列。任何没有出现的 ID 都不应传入此函数。

In [ ]:
avg_betas = ns['average_over_conditions'](standardized, conditions, ids)
X = describe('brain_features_images_by_vertices', avg_betas.T.astype(np.float32))
assert X.shape[0] == len(ids)
i = 0
repeat_values = standardized[0, conditions == ids[i]]
print('图像 ID:', ids[i], '该顶点各重复值:', repeat_values, '均值:', X[i,0])
np.testing.assert_allclose(X[i,0], repeat_values.mean(), rtol=1e-5, atol=1e-6)
raw_avg = ns['average_over_conditions'](raw[valid_vertices], conditions, ids).T
print('z-score 开关对首图前5个特征影响:')
print('不标准化:', raw_avg[0,:5]); print('按 session 标准化:', X[0,:5])

## 7. caption → 句向量 → 每图语义向量

源码 `embedding_models_zoo.py` 的 `get_embedding_model`、`get_embeddings`；这里调用其 SentenceTransformer 分支，模型路径必须是你预先准备好的本地完整模型。默认不运行，不触发模型下载。

原 `make_subj_conditional_nsd_embeddings` 对同图多个 caption 的 embedding 做均值。下面拆开这一步以观察形状，比较首句、全部均值、L2 归一化。`make_conditions_nsd_embeddings` 写死了 515 行，不适合此子集，因此不调用。

`captions` 来自仓库既有 73,000 行 caption 表，而不是虚构标注。清单固定源文件哈希；需要通过原 `NSDAccess` 重新读取时，还需额外 COCO annotation 和映射，本教学不下载那些整包文件。

In [ ]:
Y = None
if RUN_TEXT_MODEL:
    assert (LOCAL_MODEL/'modules.json').exists(), '请先准备完整本地 SentenceTransformer 模型，然后设置 LOCAL_MODEL'
    from sentence_transformers import SentenceTransformer
    emb_ns = load_functions('src/nsd_visuo_semantics/get_embeddings/embedding_models_zoo.py',
                            ['get_embedding_model','get_embeddings'], {'SentenceTransformer': SentenceTransformer})
    # 默认分支接收模型目录；避免调用在线名称自动联网下载。
    model_name = str(LOCAL_MODEL)
    model = emb_ns['get_embedding_model'](model_name)
    caption_embeddings = [np.asarray(emb_ns['get_embeddings'](caps, model, model_name)) for caps in selected.captions]
    describe('captions_of_first_image', caption_embeddings[0])
    Y = describe('semantic_images_by_dimensions', np.stack([e.mean(axis=0) for e in caption_embeddings]).astype(np.float32))
    first_caption_only = np.stack([e[0] for e in caption_embeddings])
    print('首句 vs 多句平均的平均 L2 差:', np.linalg.norm(Y-first_caption_only, axis=1).mean())
    normalized_Y = Y/np.maximum(np.linalg.norm(Y, axis=1, keepdims=True), 1e-12)
    describe('L2_normalized_semantic', normalized_Y)
    report['parameters']['text_model_path'] = str(LOCAL_MODEL)
    # 记录所有模型文件哈希，跨机器比较时路径可不同，内容应一致。
    report['model_files_sha256'] = {str(p.relative_to(LOCAL_MODEL)): hashlib.sha256(p.read_bytes()).hexdigest()
                                  for p in LOCAL_MODEL.rglob('*') if p.is_file()}
else:
    print('语义模型未运行；没有使用随机向量冒充真实 embedding。')

## 8. fMRI → 语义 embedding：小规模回归与检索

原入口 `encoding_decoding_analyses/nsd_decode_llm.py::nsd_decode_llm` 使用 `FracRidgeRegressorCV` 做脑响应→embedding；`nsd_llm_encoding_model` 则是反方向。

以下使用相同回归类，但不运行原入口（其默认 8 人×40 session、515 图、外部 GCC 候选库）。按图像 ID 划分，防止同图不同重复分到 train/test。先比较固定 `fracs`，只报告验证集指标，再在选定参数后报告测试集一次；小数据效果波动很大，检索候选只有测试集，不能与论文的大候选库指标比较。

In [ ]:
if Y is not None:
    from fracridge import FracRidgeRegressorCV
    from scipy.spatial.distance import cdist
    assert len(ids) >= 20, '样本不足，先增加教学图像数'
    order = np.random.default_rng(SEED).permutation(len(ids))
    n_train = int(len(ids)*0.6); n_val = max(5, int(len(ids)*0.2))
    train, val, test = order[:n_train], order[n_train:n_train+n_val], order[n_train+n_val:]
    # 用训练集特征统计量做第二次标准化，避免按图像行尺度差造成数值问题。
    mu, sigma = X[train].mean(axis=0), X[train].std(axis=0)
    Xreg = (X-mu)/np.maximum(sigma,1e-6)
    def retrieval(pred, truth):
        distances = cdist(pred, truth, metric='cosine')
        ranks = (np.argsort(distances, axis=1) == np.arange(len(truth))[:,None]).argmax(axis=1)+1
        return {'top1': float((ranks==1).mean()), 'median_rank': float(np.median(ranks)), 'candidates': len(truth)}
    experiments = []
    models = []
    for fraction in [0.1, 0.5, 0.9]:
        reg = FracRidgeRegressorCV(fracs=np.array([fraction]), cv=3)
        reg.fit(Xreg[train], Y[train])
        result = {'frac': fraction, **retrieval(reg.predict(Xreg[val]), Y[val])}
        experiments.append(result); models.append(reg)
    display(pd.DataFrame(experiments))
    best = max(range(len(models)), key=lambda j: experiments[j]['top1'])
    predicted = describe('predicted_test_semantic_embedding', models[best].predict(Xreg[test]))
    result = retrieval(predicted, Y[test]); print('选定参数后的测试检索:', result)
    report['validation_experiments'] = experiments
    report['test_retrieval'] = result
    report['split_ids'] = {k: ids[v].tolist() for k,v in [('train',train),('validation',val),('test',test)]}
else:
    print('先启用并运行真实 caption embedding，才能继续回归。')

## 9. 保存反馈，不需要发送大数据

只在运行到这里时创建 `OUTPUT/run_report.json`，服务器默认在 `/workspace/results/nsd_teaching_b/`。把该小文件、出错单元及完整 traceback 发给我即可；也可上传执行后的 `.ipynb`，先检查无令牌、密码或私人路径。不要推送数据、模型和大型输出到 GitHub。

下一轮可以逐项改变 session 数、顶点数量、图像尺寸、caption 均值方式和回归正则强度，其他条件不变进行对照。改变 dataset 选择需新建独立清单，不要在服务器悄悄重选。

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
report['selected_nsd_ids'] = ids.tolist()
report['stages_run'] = {'tensorflow': RUN_TENSORFLOW, 'text_model': Y is not None,
                        'full_get_betas': RUN_FULL_GET_BETAS}
report_path = OUTPUT/'run_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
print('请发给我:', report_path)